In [9]:
import pandas as pd
import numpy as np
from pandas import json_normalize
import requests
import warnings

warnings.simplefilter(action='ignore', category=FutureWarning)

df = pd.read_csv("/content/sample_data/housing_dirty.csv")

# Explorasi awal
print(df.info())
print(df.describe())
print(df.isnull().sum())
print(f"Shape awal: {df.shape}")

# Hapus duplikat
df.drop_duplicates(inplace=True)
print(f"Shape setelah hapus duplikat: {df.shape}")

# Normalisasi string
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.lower()

# Imputasi missing values
for col in df.columns:
  if df[col].dtype == 'object':
    df[col].fillna(df[col].mode()[0], inplace=True)
  else:
    df[col].fillna(df[col].median(), inplace=True)

# Tangani outlier dengan IQR Fence
for col in ['harga_juta', 'luas_m2']:
  q1, q3 = df[col].quantile([0.25, 0.75])
  IQR = q3 - q1
  df[col] = df[col].clip(q1-1.5*IQR, q3+1.5*IQR)


# Validasi Akhir
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'

# Export to csv
df.to_csv("/content/sample_data/housing_clean.csv", index=False)

# Akses API JSONPlaceholder dan simpan response sebagai DataFrame
URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL, timeout=10)

if response.status_code == 200:
  data = response.json()
  df_json = json_normalize(data, sep='_')
  print(df_json[['id', 'name', 'email', 'address_city']])
else:
  print(f'Error: {response.status_code}')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 130 entries, 0 to 129
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            130 non-null    int64  
 1   luas_m2       112 non-null    float64
 2   harga_juta    113 non-null    float64
 3   kota          130 non-null    object 
 4   kamar         120 non-null    float64
 5   tahun_bangun  130 non-null    int64  
 6   kondisi       130 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 7.2+ KB
None
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  130.000000   112.000000  1.130000e+02  120.000000    130.000000
mean    65.500000   267.627679  8.856325e+05    3.433333   2062.638462
std     37.671829   885.664181  9.407144e+06    1.776283    701.684043
min      1.000000   -50.000000 -5.000000e+02    1.000000   1890.000000
25%     33.250000    87.050000  3.450000e+02    2.000000   1991.250000
50%     65.50000